# PUMA - Stage 1: Preprocessing and training

This notebook prepares the dataset, trains the five out-of-fold Stage-1 detectors, creates OOF candidates, and trains the final Stage-1 detector. The current config uses at most 60 epochs and early-stopping patience of 15 epochs for the OOF folds. The final all-data detector runs the full schedule because it has no validation fold.

**Status: this project is still under development.**


In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Paths

Find the project folder and add its `src` directory to Python. Set `CODE_ROOT_OVERRIDE` only when the automatic path search does not find the project.


In [2]:
from pathlib import Path
import importlib.util
import subprocess
import sys

PROJECT_ROOT = Path('/content/drive/MyDrive/Research/PUMA')
CODE_ROOT_OVERRIDE = None


def is_code_root(path: Path | None) -> bool:
    return bool(
        path is not None
        and (path / 'pyproject.toml').is_file()
        and (path / 'src' / 'puma_pipeline' / '__init__.py').is_file()
    )


candidates = [
    CODE_ROOT_OVERRIDE,
    PROJECT_ROOT / 'Version 16',
    PROJECT_ROOT,
    Path.cwd(),
]
CODE_ROOT = next((p.resolve() for p in candidates if is_code_root(p)), None)
if CODE_ROOT is None:
    search_roots = [PROJECT_ROOT / 'Code', PROJECT_ROOT]
    found = []
    for parent in search_roots:
        if not parent.is_dir():
            continue
        for child in parent.iterdir():
            if child.is_dir() and is_code_root(child):
                found.append(child.resolve())
    found = sorted(set(found))
    if len(found) == 1:
        CODE_ROOT = found[0]
if CODE_ROOT is None:
    raise FileNotFoundError(
        'Cannot find the PUMA source tree. Put the extracted project under '
        f'{PROJECT_ROOT / "Code" / "PUMA"} or set CODE_ROOT_OVERRIDE.'
    )

CONFIG_TEMPLATE = CODE_ROOT / 'configs' / 'train_config.json'
if not CONFIG_TEMPLATE.is_file():
    raise FileNotFoundError(f'Missing training config: {CONFIG_TEMPLATE}')

requirements = {
    'timm': 'timm>=1.0.15,<2',
    'huggingface_hub': 'huggingface-hub>=0.23,<1',
    'tifffile': 'tifffile>=2024.2',
    'skimage': 'scikit-image>=0.22,<1',
    'scipy': 'scipy>=1.11,<2',
    'tqdm': 'tqdm>=4.66,<5',
}
missing = [spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing dependencies:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

SRC_ROOT = CODE_ROOT / 'src'
for name in list(sys.modules):
    if name == 'puma_pipeline' or name.startswith('puma_pipeline.'):
        del sys.modules[name]
if str(SRC_ROOT) in sys.path:
    sys.path.remove(str(SRC_ROOT))
sys.path.insert(0, str(SRC_ROOT))

import puma_pipeline

imported_path = Path(puma_pipeline.__file__).resolve()
if not str(imported_path).startswith(str(SRC_ROOT.resolve())):
    raise RuntimeError(f'puma_pipeline was imported from the wrong location: {imported_path}')

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CODE_ROOT    =', CODE_ROOT)
print('package      =', imported_path)


PROJECT_ROOT = /content/drive/MyDrive/Research/PUMA
CODE_ROOT    = /content/drive/MyDrive/Research/PUMA/Version 16
package      = /content/drive/MyDrive/Research/PUMA/Version 16/src/puma_pipeline/__init__.py


## 2. Install the package

Install the Python dependencies used by this project.


In [3]:
# Import the project source selected above.
print('No `pip install -e` is required for puma_pipeline.')


No `pip install -e` is required for puma_pipeline.


## 3. Load the training config

Load `configs/train_config.json` and save the resolved paths for the next notebook.


In [4]:
import json
from puma_pipeline.config import PumaConfig

RAW_IMAGE_DIR = 'Dataset/01_training_dataset_tif_ROIs'
NUCLEI_GEOJSON_DIR = 'Dataset/01_training_dataset_geojson_nuclei'
TISSUE_GEOJSON_DIR = 'Dataset/01_training_dataset_geojson_tissue'

CONFIG_TEMPLATE = CODE_ROOT / 'configs' / 'train_config.json'
if not CONFIG_TEMPLATE.is_file():
    raise FileNotFoundError(f'Missing PUMA config template: {CONFIG_TEMPLATE}')
payload = json.loads(CONFIG_TEMPLATE.read_text(encoding='utf-8'))
payload.update({
    'project_root': str(PROJECT_ROOT),
    'image_dir': RAW_IMAGE_DIR,
    'nuclei_geojson_dir': NUCLEI_GEOJSON_DIR,
    'tissue_geojson_dir': TISSUE_GEOJSON_DIR,
})
config = PumaConfig(**payload)
RESOLVED_CONFIG = CODE_ROOT / 'configs' / 'resolved_config.json'
config.save(RESOLVED_CONFIG)
print('Resolved config:', RESOLVED_CONFIG)
print('Stage-1 fingerprint:', config.stage1_fingerprint)
print('CPU workers:', config.workers)


Resolved config: /content/drive/MyDrive/Research/PUMA/Version 16/configs/resolved_config.json
Stage-1 fingerprint: f0d55e624d25ee7d
CPU workers: 10


## 4. Check the runtime

Show the current device and precision settings before preprocessing or training.


In [5]:
import os, torch

print('PyTorch:', torch.__version__)
print('CPU cores:', os.cpu_count())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('BF16 supported:', torch.cuda.is_bf16_supported())
    if config.stage1_compile and torch.cuda.get_device_capability()[0] < 8:
        print('INFO: Stage-1 torch.compile will be disabled automatically on this pre-Ampere GPU.')
    if config.use_bfloat16 and not torch.cuda.is_bf16_supported():
        print('INFO: BF16 is unavailable; Version 16 will automatically use FP16 + GradScaler.')
else:
    print('CPU-only runtime detected. This is VALID for preprocessing and dataset audit.')
    print('Switch to a GPU runtime only before Stage-1 training / Stage-1 OOF generation.')


PyTorch: 2.11.0+cu128
CPU cores: 12
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
BF16 supported: True


## 5. Preprocess the dataset

Convert the 1024x1024 TIFF ROIs and GeoJSON annotations into memory-mapped NumPy files. This step also creates the five fold assignments and the preprocessing manifest.

Set `FORCE_PREPROCESS=True` only when the preprocessing files need to be rebuilt.


In [6]:
from pprint import pprint
from puma_pipeline.data.preprocess import preprocess_dataset
from puma_pipeline.data.audit import audit_dataset

FORCE_PREPROCESS = False
preprocess_report = preprocess_dataset(config, force=FORCE_PREPROCESS)
pprint(preprocess_report)
print()
print('Dataset audit:')
print(audit_dataset(config))


{'artifact_schema': 'puma',
 'class_counts': [57351, 21668, 520, 7171, 695, 375, 3870, 1698, 2216, 1814],
 'fold_class_counts': {'0': [12013,
                             3972,
                             113,
                             1571,
                             132,
                             174,
                             741,
                             355,
                             324,
                             340],
                       '1': [11464,
                             4644,
                             108,
                             1484,
                             127,
                             50,
                             777,
                             349,
                             617,
                             326],
                       '2': [12135,
                             3940,
                             75,
                             1389,
                             187,
                             44,

## 6. Train the five Stage-1 folds

Each fold can run for up to 60 epochs. Validation runs every 5 epochs, and the best EMA checkpoint is selected by validation binary F1. Early stopping is used only for the OOF folds.


In [7]:
from puma_pipeline.stage1.trainer import train_stage1_fold

stage1_fold_summaries = []
for fold in range(config.number_of_folds):
    print(f'\n========== STAGE 1 FOLD {fold}/{config.number_of_folds - 1} ==========')
    stage1_fold_summaries.append(train_stage1_fold(config, fold, resume=True))

stage1_fold_summaries



========== STAGE 1 FOLD 0/4 ==========


Stage1 fold_0 001/60:   0%|          | 0/82 [00:00<?, ?it/s]

Stage1 fold_0 002/60:   0%|          | 0/82 [00:00<?, ?it/s]

Stage1 fold_0 003/60:   0%|          | 0/82 [00:00<?, ?it/s]

Stage1 fold_0 004/60:   0%|          | 0/82 [00:00<?, ?it/s]

...

Stage1 fold_4 057/60:   0%|          | 0/82 [00:00<?, ?it/s]

Stage1 fold_4 058/60:   0%|          | 0/82 [00:00<?, ?it/s]

Stage1 fold_4 059/60:   0%|          | 0/82 [00:00<?, ?it/s]

Stage1 fold_4 060/60:   0%|          | 0/82 [00:00<?, ?it/s]

[{'run': 'fold_0',
  'maximum_epochs': 60,
  'completed_epoch': 50,
  'selected_epoch': 35,
  'selection': 'best_validation_ema',
  'early_stopping_enabled': True,
  'early_stopping_patience': 15,
  'early_stopped': True,
  'best_validation_binary_f1': 0.8355136178425906,
  'final_checkpoint': '/content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/fold_0/stage1_final_ema.pt',
  'runtime_amp_dtype': 'bfloat16',
  'runtime_micro_batch_size': 8,
  'runtime_accumulation_steps': 1,
  'last_epoch': {'epoch': 50,
   'learning_rate': 2.407321188531556e-05,
   'train_total': 0.7542843273500117,
   'train_heatmap': 0.9267229495978937,
   'train_offset': 0.7288830709166643,
   'train_quality': 0.2511215308090536,
   'train_uncertainty': -2.017318615099279,
   'train_coarse_type': 1.032782191183509,
   'validation_binary_precision': 0.7082430284640948,
   'validation_binary_recall': 0.9935140613123892,
   'validation_binary_f1': 0.8269680929585187,
   'validation_true_positive': 19607,
   'vali

## 7. Create Stage-1 OOF predictions

Predict each ROI with the fold model that did not train on that ROI. These candidates and GT-aligned features are used by Stage 2.


In [8]:
from puma_pipeline.stage1.oof import generate_stage1_oof
from puma_pipeline.stage1 import stage1_oof_paths

FORCE_STAGE1_OOF = False
oof_metadata = generate_stage1_oof(config, force=FORCE_STAGE1_OOF)
print('OOF metadata:', oof_metadata)
print('OOF files:', stage1_oof_paths(config))


Stage1 OOF fold 0:   0%|          | 0/6 [00:00<?, ?it/s]

Stage1 OOF fold 1:   0%|          | 0/6 [00:00<?, ?it/s]

Stage1 OOF fold 2:   0%|          | 0/6 [00:00<?, ?it/s]

Stage1 OOF fold 3:   0%|          | 0/6 [00:00<?, ?it/s]

Stage1 OOF fold 4:   0%|          | 0/6 [00:00<?, ?it/s]

OOF metadata: {'artifact_schema': 'puma', 'config_fingerprint': 'f0d55e624d25ee7d', 'checkpoint_sha256': {'0': '8a19526cf42f09bbd47b6b5dc7295ea4f5682479a6a383e2948b48d7b13d684a', '1': '203050673c864d0799be5fdd9b38a4371bb032fc01fe252191dbd9a12931a392', '2': '351e59d20edff888693c6bb6370eae788b7de93151d04c501cd12e115b8b0b2a', '3': 'b93763eea01424173984791ee0a0059ee375889755dd535118460b74471d373a', '4': '785a2f0b45b4d503156a2fa9016b4c25b4cceec741b6e3a7f1c972840461ff27'}, 'number_of_candidates': 138935, 'number_of_matched_candidates': 96520, 'number_of_reject_candidates': 42415, 'number_of_gt_features': 97378, 'folds': {'0': {'rois': 41, 'candidates': 26895, 'matched': 19480, 'rejects': 7415}, '1': {'rois': 41, 'candidates': 27829, 'matched': 19799, 'rejects': 8030}, '2': {'rois': 41, 'candidates': 28886, 'matched': 19493, 'rejects': 9393}, '3': {'rois': 41, 'candidates': 27833, 'matched': 19197, 'rejects': 8636}, '4': {'rois': 41, 'candidates': 27492, 'matched': 18551, 'rejects': 8941}}, '

## 8. Train the final Stage-1 model

Train the detector on all training ROIs for the full configured 60 epochs. This model is used by the final pipeline.


In [9]:
final_stage1 = train_stage1_fold(config, None, resume=True)
final_stage1


Stage1 final_all_data 001/60:   0%|          | 0/102 [00:00<?, ?it/s]

Stage1 final_all_data 002/60:   0%|          | 0/102 [00:00<?, ?it/s]

Stage1 final_all_data 003/60:   0%|          | 0/102 [00:00<?, ?it/s]

Stage1 final_all_data 004/60:   0%|          | 0/102 [00:00<?, ?it/s]

Stage1 final_all_data 005/60:   0%|          | 0/102 [00:00<?, ?it/s]

...

Stage1 final_all_data 057/60:   0%|          | 0/102 [00:00<?, ?it/s]

Stage1 final_all_data 058/60:   0%|          | 0/102 [00:00<?, ?it/s]

Stage1 final_all_data 059/60:   0%|          | 0/102 [00:00<?, ?it/s]

Stage1 final_all_data 060/60:   0%|          | 0/102 [00:00<?, ?it/s]

{'run': 'final_all_data',
 'maximum_epochs': 60,
 'completed_epoch': 60,
 'selected_epoch': 60,
 'selection': 'last_ema',
 'early_stopping_enabled': False,
 'early_stopping_patience': None,
 'early_stopped': False,
 'best_validation_binary_f1': None,
 'final_checkpoint': '/content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/final_all_data/stage1_final_ema.pt',
 'runtime_amp_dtype': 'bfloat16',
 'runtime_micro_batch_size': 8,
 'runtime_accumulation_steps': 1,
 'last_epoch': {'epoch': 60,
  'learning_rate': 2e-06,
  'train_total': 0.7271293273159102,
  'train_heatmap': 0.8948557219084572,
  'train_offset': 0.7044346794193866,
  'train_quality': 0.24546835337783776,
  'train_uncertainty': -2.044144176969341,
  'train_coarse_type': 1.0011926608927109,
  'early_stopping_triggered': False,
  'best_validation_epoch': None,
  'best_validation_binary_f1': None}}

## 9. Check Stage-1 outputs

Confirm that the expected checkpoints and OOF files were created.


In [10]:
from puma_pipeline.stage1 import stage1_final_checkpoint, stage1_fold_checkpoint

for fold in range(config.number_of_folds):
    print(f'fold {fold}:', stage1_fold_checkpoint(config, fold))
print('final:', stage1_final_checkpoint(config))
print()
print('Stage 1 is complete. Continue with notebooks/02_Stage2_Train.ipynb')


fold 0: /content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/fold_0/stage1_final_ema.pt
fold 1: /content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/fold_1/stage1_final_ema.pt
fold 2: /content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/fold_2/stage1_final_ema.pt
fold 3: /content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/fold_3/stage1_final_ema.pt
fold 4: /content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/fold_4/stage1_final_ema.pt
final: /content/drive/MyDrive/Research/PUMA/PUMA_stage1_outputs/final_all_data/stage1_final_ema.pt

Stage 1 is complete. Continue with notebooks/02_Stage2_Train.ipynb


In [11]:
from google.colab import runtime
runtime.unassign()